# Análisis estadístico de la base de datos de Pokémon


Este análisis exploratorio tiene como objetivo caracterizar estadística y descriptivamente los Pokémon utilizables en el metajuego OU de Pokémon Showdown, atendiendo tanto a sus estadísticas base como a sus características cualitativas (tipos, habilidades, peso, altura, etc.). El análisis busca identificar patrones estructurales que ayuden a comprender su uso competitivo y sirvan como base para posteriores modelos predictivos de resultados de batalla.

### Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

datos = pd.read_csv(
    "../data/csvs/Pokemon_filtrado.csv",
    sep=",",
    encoding="utf-8",
    index_col=None
)

datos.head()


In [ ]:
# Colores por tipo y estadística
COLOR_TIPOS = {
    'Normal': '#9fa19f',
    'Lucha': '#ff8101',
    'Volador': '#81b9ef',
    'Veneno': '#9142cb',
    'Tierra': '#925122',
    'Roca': '#a9af82',
    'Bicho': '#91a119',
    'Fantasma': '#714171',
    'Acero': '#61a1b9',
    'Fuego': '#e72829',
    'Agua': '#2881f0',
    'Planta': '#3fa22a',
    'Eléctrico': '#fac000',
    'Psíquico': '#f0417a',
    'Hielo': '#3fcdf3',
    'Dragón': '#5060e2',
    'Siniestro': '#624d4f',
    'Hada': '#ef71ef'
}

COLOR_STATS = {
    'ps': '#800080',
    'ataque': '#FF0000',
    'defensa': '#008080',
    'ataque_especial': '#FFA500',
    'defensa_especial': '#0000FF',
    'velocidad': '#008000'
}


In [ ]:
datos[datos["tipo_1"].isna()]

In [ ]:
# Eliminar duplicados por nombre (por seguridad)
datos = datos.drop_duplicates(subset="nombre")

### Estadísticos descriptivos univariantes

In [ ]:
columnas_numericas = [
    "ps", "ataque", "defensa",
    "ataque_especial", "defensa_especial", "velocidad",
    "altura_m", "peso_kg"
]

# Tabla descriptiva
resumen_descriptivo = datos[columnas_numericas].describe().T
resumen_descriptivo["asimetria"] = datos[columnas_numericas].skew()

resumen_descriptivo

### Estadísticas totales

In [ ]:
# Total de estadísticas base
datos["total_estadisticas"] = (
    datos["ps"] + datos["ataque"] + datos["defensa"] +
    datos["ataque_especial"] + datos["defensa_especial"] +
    datos["velocidad"]
)

# Aguante defensivo
datos["defensiva_fisica"] = datos["ps"] + datos["defensa"]
datos["defensiva_especial"] = datos["ps"] + datos["defensa_especial"]

# Potencial ofensivo
datos["ofensiva_fisica"] = datos["ataque"]
datos["ofensiva_especial"] = datos["ataque_especial"]

# total
datos["ofensiva_total"] = datos["ataque"] + datos["ataque_especial"]
datos["defensiva_total"] = datos["defensa"] + datos["defensiva_especial"]


In [ ]:
metricas_derivadas = [
    "defensiva_fisica", "defensiva_especial",
    "ofensiva_fisica", "ofensiva_especial",
    "ofensiva_total"
]

resumen_metricas = datos[metricas_derivadas].describe().T
resumen_metricas["asimetria"] = datos[metricas_derivadas].skew()

resumen_metricas

In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(
    data=datos,
    x="ofensiva_fisica",
    y="defensiva_fisica",
    hue="tipo_1",
    palette=COLOR_TIPOS,
    alpha=0.7
)
plt.title("Ofensiva física vs defensa física")
plt.xlabel("Ofensiva física")
plt.ylabel("Defensa física")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Tipo 1')
plt.show()

In [ ]:
plt.figure(figsize=(7,7))
sns.scatterplot(
    data=datos,
    x="ofensiva_especial",
    y="defensiva_especial",
    hue="tipo_1",
    palette=COLOR_TIPOS,
    alpha=0.7
)
plt.title("Ofensiva especial vs defensa especial")
plt.xlabel("Ofensiva especial")
plt.ylabel("Defensa especial")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Tipo 1')
plt.show()

### Distribución de estadísticas base

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,6))

for stat, color in zip(
    ["ps", "ataque", "defensa", "ataque_especial", "defensa_especial", "velocidad"],
    COLOR_STATS.values()
):
    sns.kdeplot(
        datos[stat],
        label=stat.replace("_", " ").title(),
        color=color,
        linewidth=2
    )

plt.title("Distribución de estadísticas base")
plt.xlabel("Valor de la estadística")
plt.ylabel("Densidad")
plt.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14,8))

stats = ["ps", "ataque", "defensa", "ataque_especial", "defensa_especial", "velocidad"]

for ax, stat in zip(axes.flatten(), stats):
    sns.histplot(datos[stat], kde=True, ax=ax)
    ax.set_title(stat.replace("_", " ").title())
    ax.set_xlabel("Valor de la estadística")

plt.tight_layout()
plt.show()

### Análisis específico de la velocidad

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(
    datos["velocidad"],
    bins=30,
    kde=True,
    color=COLOR_STATS["velocidad"]
)
plt.title("Distribución de la velocidad")
plt.xlabel("Velocidad")
plt.ylabel("Frecuencia")
plt.show()


In [ ]:
# Cuantiles relevantes
datos["velocidad"].quantile([0.25, 0.5, 0.75, 0.9, 0.95])

### Análisis por tipo (frecuencia)

In [ ]:
frecuencia_tipos = datos["tipo_1"].value_counts()

plt.figure(figsize=(12,6))
sns.barplot(
    x=frecuencia_tipos.index,
    y=frecuencia_tipos.values,
    palette=[COLOR_TIPOS[t] for t in frecuencia_tipos.index]
)

plt.title("Frecuencia de Tipo 1")
plt.xlabel("Tipo")
plt.ylabel("Número de Pokémon")
plt.xticks(rotation=45)
plt.show()


### Estadísticas medias por tipo

In [ ]:
estadisticas_por_tipo = datos.groupby("tipo_1")[
    ["ps", "ataque", "defensa",
     "ataque_especial", "defensa_especial", "velocidad"]
].mean()


In [ ]:
estadisticas_por_tipo.plot(
    kind="bar",
    figsize=(14,6),
    color=[COLOR_STATS["ps"],
           COLOR_STATS["ataque"],
           COLOR_STATS["defensa"],
           COLOR_STATS["ataque_especial"],
           COLOR_STATS["defensa_especial"],
           COLOR_STATS["velocidad"]]
)

plt.title("Estadísticas medias por Tipo 1")
plt.xlabel("Tipo")
plt.ylabel("Valor medio")
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1, 1), loc='upper left', title='Estadística')
plt.show()

### Número de habilidades

In [ ]:
datos["numero_habilidades"] = datos[
    ["habilidad_1", "habilidad_2", "habilidad_3"]
].notna().sum(axis=1)

datos["numero_habilidades"].value_counts(normalize=True)


In [ ]:
datos.head()

### Normalidad

In [ ]:
from scipy.stats import shapiro
# Seleccionar variables numéricas
variables_numericas = datos.select_dtypes(include=['int64', 'float64']).columns

# Lista para guardar resultados
resultados_normalidad = []

# Aplicar Shapiro-Wilk a cada variable
for var in variables_numericas:

    # Extraer columna y eliminar NA
    muestra = datos[var].dropna()

    # Si hay demasiados datos, tomar muestra
    if len(muestra) > 5000:
        muestra = muestra.sample(5000, random_state=42)

    # Test de Shapiro-Wilk
    stat, p = shapiro(muestra)

    # Guardar resultados
    resultados_normalidad.append({
        "Variable": var,
        "Estadistico_W": stat,
        "p_valor": p,
        "Normalidad": "Sí" if p > 0.05 else "No"
    })

# Convertir a DataFrame
tabla_normalidad = pd.DataFrame(resultados_normalidad)

# Ordenar resultados
tabla_normalidad = tabla_normalidad.sort_values("p_valor")

# Mostrar tabla
print(tabla_normalidad)

### Correlaciones entre estadísticas

In [ ]:
matriz_correlacion = datos[
    ["ps", "ataque", "defensa", "numero_habilidades","ofensiva_fisica","ofensiva_especial","ofensiva_total","defensiva_total",
     "ataque_especial", "defensa_especial", "velocidad",
     "total_estadisticas", "defensiva_fisica", "defensiva_especial", "peso_kg", "altura_m"]
].corr()

plt.figure(figsize=(12,10))
sns.heatmap(
    matriz_correlacion,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Matriz de correlación de las estadísticas base")
plt.show()


In [ ]:
print(matriz_correlacion)

### Gráficos de combinaciones de tipos

In [ ]:
import networkx as nx

datos[['tipo_1', 'tipo_2']]
matrix = datos.groupby(['tipo_1', 'tipo_2']).size().unstack(fill_value=0)
matrix = matrix[COLOR_TIPOS.keys()].loc[COLOR_TIPOS.keys()]

# Crear grafo
G = nx.Graph()

# Añadir nodos
for tipo in matrix.index:
    G.add_node(tipo)

# Añadir aristas con pesos
for i, tipo1 in enumerate(matrix.index):
    for j, tipo2 in enumerate(matrix.columns):
        if i < j:  # Para evitar duplicados en grafo no dirigido
            weight = matrix.iloc[i, j]
            if weight > 0:
                G.add_edge(tipo1, tipo2, weight=weight)

# Dibujar el grafo
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, seed=42)  # Layout para posicionar nodos
edges = G.edges()
weights = [G[u][v]['weight'] for u, v in edges]

nx.draw(G, pos, with_labels=True, node_color=[COLOR_TIPOS[node] for node in G.nodes()], 
        node_size=1000, font_size=10, font_weight='bold', 
        width=[w*0.1 for w in weights], edge_color='gray', alpha=0.7)

# Añadir etiquetas de peso en las aristas
edge_labels = {(u, v): G[u][v]['weight'] for u, v in edges if G[u][v]['weight'] > 0}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8)

plt.title("Grafo de combinaciones de tipos de Pokémon")
plt.axis('off')
plt.show()

In [ ]:
from d3blocks import D3Blocks

# Crear source-target-weight
links = []

for source in adjacency_matrix.index:
    for target in adjacency_matrix.columns:

        value = adjacency_matrix.loc[source, target]

        if value > 0:
            links.append([
                source,
                target,
                value
            ])

# DataFrame final
df_links = pd.DataFrame(
    links,
    columns=['source', 'target', 'weight']
)

# Inicializar chord diagram
d3 = D3Blocks(chart='Chord', frame=False)

# Crear nodos
d3.set_node_properties(
    df_links,
    opacity=0.9
)

# Aplicar tus colores
for tipo, color in COLOR_TIPOS.items():

    if tipo in d3.node_properties:
        d3.node_properties[tipo]['color'] = color

# Propiedades edges
d3.set_edge_properties(
    df_links,
    color='source',
    opacity='source'
)

# Mostrar gráfico
d3.show()

Mapa de calor

In [ ]:
tipos = list(COLOR_TIPOS.keys())
matriz = (
    datos
    .groupby(["tipo_1", "tipo_2"])
    .size()
    .unstack(fill_value=0)
)

# Reordenar filas y columnas
matriz = matriz.reindex(index=tipos, columns=tipos, fill_value=0)
matriz_simetrica = matriz + matriz.T
for tipo in matriz.index:
    matriz_simetrica.loc[tipo, tipo] = matriz.loc[tipo, tipo]

plt.figure(figsize=(13, 11))

sns.heatmap(
    matriz_simetrica,
    annot=True,              # ← números en cada celda
    fmt="d",                 # ← enteros
    cmap="YlOrRd",
    linewidths=0.5,
    square=True,
    cbar_kws={"label": "Número de Pokémon"}
)

plt.title(
    "Mapa de calor de combinaciones de tipos en Pokémon OU",
    fontsize=16,
    pad=20
)

plt.xlabel("Tipo 1")
plt.ylabel("Tipo 2")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()


### Top 10 Pokémon por estadística

In [ ]:
estadisticas = {
    "ps": "ps",
    "ataque": "ataque",
    "defensa": "defensa",
    "ataque_especial": "ataque_especial",
    "defensa_especial": "defensa_especial",
    "velocidad": "velocidad"
}
def top_10_por_estadistica(datos, columna, nombre_legible):
    top10 = (
        datos[["nombre", columna]]
        .sort_values(by=columna, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    
    print(f"Top 10 Pokémon por {nombre_legible}")
    display(top10)
    
    return top10
tops = {}

for col, nombre in estadisticas.items():
    tops[col] = top_10_por_estadistica(datos, col, nombre)


In [ ]:
def grafico_top10(datos, columna, nombre_legible):
    top10 = (
        datos[["nombre", columna]]
        .sort_values(by=columna, ascending=False)
        .head(10)
    )

    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=top10,
        x=columna,
        y="nombre",
        color=COLOR_STATS[nombre_legible]
    )

    plt.title(f"Top 10 Pokémon por {nombre_legible}", fontsize=14)
    plt.xlabel(nombre_legible)
    plt.ylabel("Pokémon")

    plt.tight_layout()
    plt.show()


In [ ]:
for col, nombre in estadisticas.items():
    grafico_top10(datos, col, nombre)


In [ ]:
datos.head()

In [ ]:
datos.info()

### Análisis de Componentes principales

In [ ]:
print(datos["tipo_1"].unique())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

# Variables numericas para el PCA

num_features = [
    'ps', 'ataque', 'defensa',
    'ataque_especial', 'defensa_especial', 'velocidad',
    'altura_m', 'peso_kg',
    'defensiva_fisica', 'defensiva_especial',
    'ofensiva_fisica', 'ofensiva_especial',
    'ofensiva_total', 'defensiva_total',
    'numero_habilidades'
]

# Filtrar filas validas

mask = datos[num_features].notna().all(axis=1)
datos_pca = datos.loc[mask].copy()

print('Numero de Pokemon usados en PCA:', datos_pca.shape[0])

# Escalado
X = datos_pca[num_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Colores por tipo
colores = datos_pca['tipo_1'].map(COLOR_TIPOS)
print('Colores NaN:', colores.isna().sum())

# PCA en 2 componentes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('Varianza explicada por componente:', pca.explained_variance_ratio_)
print('Varianza acumulada:', pca.explained_variance_ratio_.sum())

# Grafico PCA 2D coloreado por tipo
plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colores,
    s=30,
    alpha=0.7
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA 2D de la Pokédex coloreado por Tipo 1')
plt.grid(alpha=0.3)

# Leyenda manual de tipos
tipos_presentes = datos_pca['tipo_1'].unique()

legend_elements = [
    Patch(facecolor=COLOR_TIPOS[tipo], label=tipo)
    for tipo in tipos_presentes
    if tipo in COLOR_TIPOS
]

plt.legend(
    handles=legend_elements,
    title='Tipo 1',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import Patch

# Variables numericas para el PCA

num_features = [
    'ps', 'ataque', 'defensa',
    'ataque_especial', 'defensa_especial', 'velocidad',
    'altura_m', 'peso_kg',
    'defensiva_fisica', 'defensiva_especial',
    'ofensiva_fisica', 'ofensiva_especial',
    'ofensiva_total',
    'numero_habilidades'
]

# Filtrar filas validas

mask = datos[num_features].notna().all(axis=1)
datos_pca = datos.loc[mask].copy()

print('Numero de Pokemon usados en PCA:', datos_pca.shape[0])

# Escalado

X = datos_pca[num_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Colores por tipo

colores = datos_pca['tipo_1'].map(COLOR_TIPOS)
print('Colores NaN:', colores.isna().sum())

# PCA en 3 componentes

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print('Varianza explicada por componente:', pca.explained_variance_ratio_)
print('Varianza acumulada:', pca.explained_variance_ratio_.sum())

# Grafico PCA 3D coloreado por tipo

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    X_pca[:, 2],
    c=colores,
    s=30,
    alpha=0.7
)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('PCA 3D')

# Leyenda manual de tipos

tipos_presentes = datos_pca['tipo_1'].unique()

legend_elements = [
    Patch(facecolor=COLOR_TIPOS[tipo], label=tipo)
    for tipo in tipos_presentes
    if tipo in COLOR_TIPOS
]

ax.legend(
    handles=legend_elements,
    title='Tipo 1',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()


#### T-SNE

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

tsne = TSNE(
    n_components=2,
    perplexity=50,
    learning_rate=200,
    max_iter=500,
    random_state=42
)

X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    c=colores,
    s=30,
    alpha=0.7
)

plt.xlabel('Dim 1')
plt.ylabel('Dim 2')
plt.title('t-SNE de la Pokédex coloreado por Tipo 1')
plt.grid(alpha=0.3)

# Leyenda
tipos_presentes = datos['tipo_1'].unique()
legend_elements = [
    Patch(facecolor=COLOR_TIPOS[tipo], label=tipo)
    for tipo in tipos_presentes
    if tipo in COLOR_TIPOS
]

plt.legend(
    handles=legend_elements,
    title='Tipo 1',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()


#### UMAP

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import umap.umap_ as umap
from matplotlib.patches import Patch

# Variables numericas

num_features = [
    'ps', 'ataque', 'defensa',
    'ataque_especial', 'defensa_especial', 'velocidad',
    'altura_m', 'peso_kg',
    'defensiva_fisica', 'defensiva_especial',
    'ofensiva_fisica', 'ofensiva_especial',
    'ofensiva_total',
    'numero_habilidades'
]

# Filtrar filas validas

mask = datos[num_features].notna().all(axis=1)
datos_red = datos.loc[mask].copy()

print('Numero de Pokemon usados:', datos_red.shape[0])

# Escalado

X = datos_red[num_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Colores por tipo

colores = datos_red['tipo_1'].map(COLOR_TIPOS)
print('Colores NaN:', colores.isna().sum())

# UMAP

umap_model = umap.UMAP(
    n_neighbors=30, 
    min_dist=0.1,
    n_components=2,
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)

# Grafico UMAP

plt.figure(figsize=(8, 6))
plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=colores,
    s=30,
    alpha=0.7
)

plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('UMAP de la Pokédex coloreado por Tipo 1')
plt.grid(alpha=0.3)

# Leyenda
tipos_presentes = datos_red['tipo_1'].unique()

legend_elements = [
    Patch(facecolor=COLOR_TIPOS[tipo], label=tipo)
    for tipo in tipos_presentes
    if tipo in COLOR_TIPOS
]

plt.legend(
    handles=legend_elements,
    title='Tipo 1',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()
